## Forecast 1 hari ke depan — oil, water, choke

**Ide:** Di akhir hari \(t\) kamu anggap sudah tahu produksi & operasi hari \(t\). Model memprediksi **oil, water, dan choke** untuk **hari \(t+1\)**.

**Catatan choke:** Choke besok sering diputus operator; model ini mempelajari **pola historis** (bukan aturan fisik). Bandingkan metriknya dengan baseline (mis. choke besok = choke hari ini).

**Anti-leakage:** Target dibuat dengan `shift(-1)`. Fitur rolling memakai `shift(1)` sebelum `rolling` agar tidak memakai info “masa depan”.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error
import lightgbm as lgb

### 1. Muat data (CSV atau Excel)
Sesuaikan path jika pakai file `.xlsx` seperti notebook lain.

In [ ]:
# Ganti ke read_excel(..., engine='openpyxl') jika perlu
df = pd.read_csv("data/volve_production_data.csv", parse_dates=["DATEPRD"])

# Sama seperti praproses di web / notebook utama
df = df[df["BORE_OIL_VOL"] > 0]
df = df[df["FLOW_KIND"].astype(str).str.lower() == "production"]
df = df[df["WELL_TYPE"].astype(str) == "OP"]

WELL = "15/9-F-14"   # ganti sumur di sini
CHOKE_COL = "AVG Choke size"

d = df[df["NPD_WELL_BORE_NAME"] == WELL].copy()
d = d.sort_values("DATEPRD").reset_index(drop=True)
print(WELL, "baris:", len(d))
d.head(3)

### 2. Target = nilai **besok** (1 hari ke depan)

In [ ]:
d["y_oil"] = d["BORE_OIL_VOL"].shift(-1)
d["y_water"] = d["BORE_WAT_VOL"].shift(-1)
d["y_choke"] = d[CHOKE_COL].shift(-1)

# Baris terakhir tidak punya "besok" → buang
d = d.dropna(subset=["y_oil", "y_water", "y_choke"]).reset_index(drop=True)
len(d)

### 3. Fitur di hari \(t\) (lag + rolling aman + operasi hari ini)

In [ ]:
oil, wat, ch = d["BORE_OIL_VOL"], d["BORE_WAT_VOL"], d[CHOKE_COL]

d["oil_lag1"] = oil.shift(1)
d["water_lag1"] = wat.shift(1)
d["choke_lag1"] = ch.shift(1)
d["oil_lag4"] = oil.shift(4)
d["water_lag4"] = wat.shift(4)
d["choke_lag4"] = ch.shift(4)
d["oil_lag7"] = oil.shift(7)
d["water_lag7"] = wat.shift(7)

# Rolling hanya dari hari-hari SEBELUM t (shift dulu)
d["oil_roll7"] = oil.shift(1).rolling(7, min_periods=1).mean()
d["water_roll7"] = wat.shift(1).rolling(7, min_periods=1).mean()

ops_cols = [
    "ON_STREAM_HRS",
    "AVG_DOWNHOLE_PRESSURE",
    "AVG_DOWNHOLE_TEMPERATURE",
    "AVG_DP_TUBING",
    "DP_CHOKE_SIZE",
    "AVG_WHP_P",
    "AVG_WHT_P",
    CHOKE_COL,
    "BORE_OIL_VOL",
    "BORE_WAT_VOL",
]

feat_cols = [
    "oil_lag1", "water_lag1", "choke_lag1",
    "oil_lag4", "water_lag4", "choke_lag4",
    "oil_lag7", "water_lag7",
    "oil_roll7", "water_roll7",
] + [c for c in ops_cols if c in d.columns]

d = d.dropna(subset=feat_cols).reset_index(drop=True)
print("Baris setelah drop NaN fitur:", len(d))
feat_cols

### 4. Split **berdasarkan waktu** (bukan random)
Misal 80% awal = train, 20% akhir = test.

In [ ]:
frac_train = 0.8
n = len(d)
cut = int(n * frac_train)

train = d.iloc[:cut].copy()
test = d.iloc[cut:].copy()

X_train = train[feat_cols]
X_test = test[feat_cols]

print("Train:", len(train), "| Test:", len(test))
print("Test DATEPRD:", test["DATEPRD"].min().date(), "→", test["DATEPRD"].max().date())

### 5. Tiga model LightGBM (oil, water, choke)
Satu model per target — sederhana dan mudah di-debug.

In [ ]:
def metrics(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    print(f"{name:12}  MAE={mae:.4f}  RMSE={rmse:.4f}")
    return mae, rmse


def fit_lgbm(X_tr, y_tr, X_te, y_te, label):
    model = lgb.LGBMRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=31,
        random_state=42,
        verbose=-1,
    )
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    metrics(y_te, pred, label)
    return model, pred


m_oil, pred_oil = fit_lgbm(X_train, train["y_oil"], X_test, test["y_oil"], "Oil")
m_wat, pred_wat = fit_lgbm(X_train, train["y_water"], X_test, test["y_water"], "Water")
m_cho, pred_cho = fit_lgbm(X_train, train["y_choke"], X_test, test["y_choke"], "Choke")

### 6. Baseline choke (opsional): besok = hari ini
Kalau MAE model choke **lebih besar** dari baseline, berarti pola choke sulit dipelajari dari data saja.

In [ ]:
naive_choke = test[CHOKE_COL].values
metrics(test["y_choke"], naive_choke, "Choke naive")

### 7. Plot aktual vs prediksi (periode test)

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
x = range(len(test))

ax[0].plot(x, test["y_oil"].values, label="Aktual", color="gray")
ax[0].plot(x, pred_oil, label="Prediksi", color="green", alpha=0.85)
ax[0].set_ylabel("Oil")
ax[0].legend()
ax[0].set_title(f"Forecast 1 hari — {WELL} (test)")

ax[1].plot(x, test["y_water"].values, label="Aktual", color="gray")
ax[1].plot(x, pred_wat, label="Prediksi", color="blue", alpha=0.85)
ax[1].set_ylabel("Water")
ax[1].legend()

ax[2].plot(x, test["y_choke"].values, label="Aktual", color="gray")
ax[2].plot(x, pred_cho, label="Prediksi", color="teal", alpha=0.85)
ax[2].plot(x, naive_choke, label="Naive (choke hari ini)", color="orange", alpha=0.6, linestyle="--")
ax[2].set_ylabel("Choke")
ax[2].legend()

plt.xlabel("Index urutan waktu (test)")
plt.tight_layout()
plt.show()

### Ringkasan alur (untuk nanti di web)
1. Data per sumur, urut `DATEPRD`.
2. Target `y_*` = `shift(-1)` untuk oil, water, choke.
3. Fitur = lag + rolling (aman) + operasi & produksi hari \(t\).
4. Split **kronologis**.
5. Tiga regressor (atau multi-output).
6. Evaluasi di **test akhir** + bandingkan baseline choke.

Kalau metrik di test sudah oke, barulah porting ke Flask dengan endpoint serupa `/api/forecast`.